In [1]:
COMPUTATIONAL_METRICS = ['duration_seconds', 'inference_time', 'mean_leaves', 'mean_nodes', 'ntrees',
        'train_time', 'get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']
DETAILED_COMPUTATIONAL = ['get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']

In [3]:

TO_INT = ['sketch_outputs']
TO_FLOAT = ['smoothing_alpha', 'stabilization_threshold', 'subsample', 'lr', ]

In [4]:
import os 

os.getcwd() 

'/home/leostre/Рабочий стол/py-boost/ablations'

In [5]:
os.chdir('../5.2')

In [6]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def get_all_runs_data(experiment_names=None, include_artifacts=False):
    """
    Extract all runs data from MLflow experiments into a comprehensive DataFrame
    
    Args:
        experiment_names: List of experiment names or None for all experiments
        include_artifacts: Whether to include artifact URIs
    
    Returns:
        DataFrame with all runs data
    """
    client = MlflowClient()
    
    # Get experiments
    if experiment_names is None:
        experiments = client.search_experiments()
    else:
        experiments = [client.get_experiment_by_name(name) for name in experiment_names]
        experiments = [exp for exp in experiments if exp is not None]
    
    all_runs_data = []
    
    for experiment in tqdm(experiments, desc="Processing experiments"):
        experiment_id = experiment.experiment_id
        experiment_name = experiment.name
        
        print(f"Processing experiment: {experiment_name}")
        
        # Get all runs for this experiment
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            max_results=10000  # Adjust if you have more runs
        )
        
        for run in tqdm(runs, desc=f"Runs in {experiment_name}", leave=False):
            run_data = {
                'run_id': run.info.run_id,
                'experiment_id': experiment_id,
                'experiment_name': experiment_name,
                'run_name': run.data.tags.get('mlflow.runName', ''),
                'status': run.info.status,
                'start_time': pd.to_datetime(run.info.start_time, unit='ms'),
                'end_time': pd.to_datetime(run.info.end_time, unit='ms') if run.info.end_time else None,
                'duration_seconds': (run.info.end_time - run.info.start_time) / 1000.0 if run.info.end_time and run.info.start_time else None,
            }
            
            # Add parameters
            for key, value in run.data.params.items():
                run_data[f'param_{key}'] = value
            
            # Add metrics
            for key, value in run.data.metrics.items():
                run_data[f'metric_{key}'] = value
            
            # Add tags
            for key, value in run.data.tags.items():
                if key not in ['mlflow.runName', 'mlflow.user']:
                    run_data[f'tag_{key}'] = value
            
            # Add artifact location if requested
            if include_artifacts:
                run_data['artifact_uri'] = run.info.artifact_uri
            
            all_runs_data.append(run_data)
    
    return pd.DataFrame(all_runs_data)

def get_baselines(path, version):
    start_dir = os.getcwd()
    os.chdir(path)
    df = get_all_runs_data(['baselines_' + str(version)])
    os.chdir(start_dir)
    return df 


def filter_df(df, rule):
    for k, v in rule.items():
        if k not in df.columns:
            continue
        df = df[df[k] == v]
    return df

In [7]:
EXCLUDE = {'experiment_id', 'experiment_name', 'status', 'start_time', 'end_time', 'run_name', 'tag_mlflow.source.name', 'tag_mlflow.source.git.commit',
       'tag_mlflow.source.type', 'param_error', 'tag_status',
       'param_total_runs', 'param_successful_runs', 'mean_f1', 'mean_accuracy', 'param_n_splits', 'param_n_successful_folds'}

def filter_data(data):
    after_exclusion_by_name = [
        col for col in data.columns if col not in EXCLUDE
    ]
    print(after_exclusion_by_name)
    statistics = ('mean', 'max', 'min', 'std', 'median')
    after_exclusion_agg = [
        col for col in after_exclusion_by_name if 'leaves' in col or 'nodes' in col or
        'tree' in col or
        not any(statistic in col for statistic in statistics) and not 'metric_fold' in col or col in ('param_stabilization_threshold', 'param_smoothing_alpha')
    ]
    filtered_data = data[after_exclusion_agg 
                        #  + ['metric_std_num_trees', 'metric_mean_num_trees',]
                         ]
    filtered_pivot = filtered_data.rename(columns={col: col.removeprefix('param_') for col in filtered_data.columns})
    return filtered_pivot

def melt_metrics(df):
    # Identify metric columns
    metric_cols = [col for col in df.columns if col.startswith('metric_')]
    print(metric_cols)
    
    # Identify ID columns (all non-metric columns)
    id_cols = [col for col in df.columns if not col.startswith('metric_')]
    print(id_cols)
    
    # Melt using pandas melt (more control)
    melted_df = df.melt(
        id_vars=id_cols,
        value_vars=metric_cols,
        var_name='metric_fold',
        value_name='value'
    )
    
    # Extract metric name and fold number using regex pattern
    pattern = r'metric_(.+?)_fold_(\d+)$'
    extracted = melted_df['metric_fold'].str.extract(pattern)
    
    # Create new columns
    melted_df['metric'] = extracted[0]
    melted_df['fold'] = extracted[1]
    
    # For metrics without fold numbers (like 'metric_total_training_time')
    # Fill NaN metric names with the original string without 'metric_' prefix
    mask = melted_df['metric'].isna()
    melted_df.loc[mask, 'metric'] = melted_df.loc[mask, 'metric_fold'].str.replace('metric_', '')
    
    # Drop the temporary column and clean up
    melted_df = melted_df.drop('metric_fold', axis=1)
    melted_df = melted_df.reset_index(drop=True)
    
    return melted_df


In [8]:
DATASET = [
    # 'mediamill',
    'mnist',
    'cifar10',
    # 'yeast', 
        #    'age_prediction'
        #    'birds', 
        #    'genbase'
    # 'mbd',
]

EXPS = [
    # 'mnist', 
    'sigmoid_5.2',
    'hyperbolic_5.2',
    # 'sigmoid_5.2',
]
TO_INT = ['sketch_outputs']
TO_FLOAT = ['smoothing_alpha', 'stabilization_threshold', 'subsample', 'lr', ]

dfs = [get_all_runs_data([exp]) for exp in EXPS] 

version = '5.2'
if version:
    dfs += [get_baselines(f'../{version}', version)]
    EXPS += [f'baselines_{version}']

processed = {}
for exp, df in zip(EXPS, dfs):

    df = df[df.param_dataset.isin(DATASET)]
    fp_df = filter_data(df)
    mlt_df = melt_metrics(fp_df)
    agg_mtrs = mlt_df.groupby(['dataset',
                                'sketch_method', 'sketch_outputs', 
                                *(['smoothing_alpha', 'stabilization_threshold'] if 'smoothing_alpha' in mlt_df.columns else []),
                                'subsample', 'lr', 'metric', ]).agg({'value': 'mean'}).reset_index()
    for c in TO_FLOAT:
        if not c in agg_mtrs:
            continue
        agg_mtrs[c] = agg_mtrs[c].astype(float)
    for c in TO_INT:
        if not c in agg_mtrs:
            continue
        agg_mtrs[c] = agg_mtrs[c].astype(int)
    
    processed[exp] = (agg_mtrs)

Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: sigmoid_5.2


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: hyperbolic_5.2


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: baselines_5.2


Processing experiments: 100%|██████████| 1/1 [00:00<00:00, 14.11it/s]

['run_id', 'duration_seconds', 'param_es', 'param_max_bin', 'param_dataset', 'param_sketch_params', 'param_ntrees', 'param_lr', 'param_colsample', 'param_min_data_in_bin', 'param_subsample', 'param_min_gain_to_split', 'param_stabilization_threshold', 'param_sketch_method', 'param_callbacks', 'param_gd_steps', 'param_lambda_l2', 'param_quantization', 'param_use_hess', 'param_verbose', 'param_sketch_outputs', 'param_loss', 'param_smoothing_alpha', 'param_min_data_in_leaf', 'param_quant_sample', 'param_max_depth', 'param_metric', 'param_seed', 'metric_get_indexers_calls_fold_2', 'metric_train_time_fold_3', 'metric_f1_fold_2', 'metric_train_time_fold_2', 'metric_get_weights_total_time_fold_1', 'metric_recall_fold_0', 'metric_get_weights_calls_fold_3', 'metric_inference_time_fold_2', 'metric_get_indexers_total_time_fold_3', 'metric_ntrees_fold_2', 'metric_get_indexers_avg_time_fold_1', 'metric_ntrees_fold_0', 'metric_get_weights_avg_time_fold_0', 'metric_precision_fold_0', 'metric_mean_node

In [9]:
all_exps = []
for exp, df in processed.items():
    df['experiment'] = exp
    all_exps.append(df)
all_exps = pd.concat(all_exps, axis=0)

In [10]:
INCLUDE_QUALITY = True

In [11]:
comp_metrics = all_exps[all_exps.metric.isin(COMPUTATIONAL_METRICS + (['f1'] if INCLUDE_QUALITY else []))]

In [12]:
comp_metrics.metric.unique()

array(['f1', 'get_indexers_calls', 'get_indexers_total_time',
       'get_weights_avg_time', 'get_weights_calls', 'inference_time',
       'mean_leaves', 'mean_nodes', 'ntrees', 'train_time'], dtype=object)

In [13]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from ipywidgets import interact, widgets, VBox, HBox, Output
from IPython.display import display, clear_output

def create_interactive_metric_scatter(df, x_metric, y_metric, signature_flag=False):
    """
    Create an interactive scatter plot with filtering widgets and flexible coloration.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format
    x_metric, y_metric : str
        Metrics to plot
    signature_flag : bool
        Add zero lines or not
    """
    
    # Pivot the data
    df = df.copy()
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Identify categorical columns for filtering
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    
    # Create filters dictionary
    filters = {}
    
    def create_filter_widget(col):
        """Create a filter widget for a column"""
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        return widgets.SelectMultiple(
            options=unique_vals,
            description=col,
            layout=widgets.Layout(width='300px'),
            style={'description_width': 'initial'}
        )
    
    # Create filter widgets
    filter_widgets = {col: create_filter_widget(col) for col in categorical_cols}
    
    # Create color by selector
    color_by_widget = widgets.Dropdown(
        options=['None'] + categorical_cols + ['color_category_auto'],
        value='color_category_auto',
        description='Color by:',
        layout=widgets.Layout(width='300px'),
        style={'description_width': 'initial'}
    )
    
    # Create output widget for the plot
    output_widget = Output()
    
    def update_plot(change=None):
        """Update function called when filters or color selection changes"""
        with output_widget:
            clear_output(wait=True)
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Determine coloring
            color_by = color_by_widget.value
            
            if color_by == 'None':
                # All points same color
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric}
                )
                fig.update_traces(marker=dict(color='blue', opacity=0.7))
            elif color_by == 'color_category_auto':
                # Color by unique combination of all categorical columns
                filtered_df['color_category'] = filtered_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color='color_category',
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric, 'color_category': 'Configuration'}
                )
            else:
                # Color by specific column
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color=color_by,
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric, color_by: color_by},
                    color_continuous_scale='Viridis' if filtered_df[color_by].dtype in ['float64', 'int64'] else 'Set1'
                )
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout
            fig.update_layout(
                plot_bgcolor='white',
                xaxis=dict(
                    title=x_metric,
                    gridcolor='lightgray',
                    showgrid=True,
                    zeroline=False
                ),
                yaxis=dict(
                    title=y_metric,
                    gridcolor='lightgray',
                    showgrid=True,
                    zeroline=False
                ),
                hovermode='closest',
                height=600
            )
            
            fig.show()
    
    # Create filter UI
    filter_controls = []
    
    # Group filters in rows of 3
    filter_items = list(filter_widgets.items())
    for i in range(0, len(filter_items), 3):
        row = HBox([widget for _, widget in filter_items[i:i+3]])
        filter_controls.append(row)
    
    # Create control panel
    control_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *filter_controls,
        widgets.HTML("<br><b>Styling Options:</b>"),
        HBox([color_by_widget]),
        widgets.HTML("<br><b>Plot Controls:</b>"),
        widgets.Button(description='Reset All Filters', button_style='warning')
    ])
    
    # Reset button functionality
    reset_button = control_panel.children[-1]
    def reset_filters(b):
        for widget in filter_widgets.values():
            widget.value = []
        color_by_widget.value = 'color_category_auto'
    reset_button.on_click(reset_filters)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display the complete interface
    display(VBox([control_panel, output_widget]))
    
    return filter_widgets, color_by_widget


def create_advanced_interactive_scatter(df, x_metric, y_metric, signature_flag=False):
    """
    More advanced version with additional features like:
    - Multiple color modes (categorical, numerical, custom)
    - Size encoding
    - Opacity control
    - Show/hide legend
    - Export data button
    """
    
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Get all columns for selection
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    numerical_cols = [col for col in id_vars if df[col].dtype in ['float64', 'int64'] and col not in categorical_cols]
    all_metrics = [col for col in df_pivoted.columns if col not in id_vars]
    
    # Create widgets
    filters = {}
    
    # Create filter widgets with multi-select
    filter_widgets = {}
    for col in categorical_cols:
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        filter_widgets[col] = widgets.SelectMultiple(
            options=unique_vals,
            description=col[:15],
            layout=widgets.Layout(width='250px'),
            style={'description_width': 'initial'}
        )
    
    # Color by widget
    color_options = ['None', 'Auto (All columns)'] + categorical_cols + numerical_cols + all_metrics
    color_by_widget = widgets.Dropdown(
        options=color_options,
        value='Auto (All columns)',
        description='Color by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Size by widget
    size_by_widget = widgets.Dropdown(
        options=['None'] + numerical_cols + all_metrics,
        value='None',
        description='Size by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Opacity slider
    opacity_slider = widgets.FloatSlider(
        value=0.7,
        min=0.1,
        max=1.0,
        step=0.05,
        description='Opacity:',
        layout=widgets.Layout(width='250px')
    )
    
    # Point size slider
    point_size_slider = widgets.IntSlider(
        value=8,
        min=2,
        max=20,
        description='Point size:',
        layout=widgets.Layout(width='250px')
    )
    
    # Legend toggle
    legend_toggle = widgets.Checkbox(
        value=True,
        description='Show legend',
        layout=widgets.Layout(width='150px')
    )
    
    # Grid toggle
    grid_toggle = widgets.Checkbox(
        value=True,
        description='Show grid',
        layout=widgets.Layout(width='150px')
    )
    
    # Color scale for numerical data
    color_scale_widget = widgets.Dropdown(
        options=['Viridis', 'Plasma', 'Inferno', 'Magma', 'Cividis', 'Blues', 'Reds', 'Greens'],
        value='Viridis',
        description='Color scale:',
        layout=widgets.Layout(width='250px')
    )
    
    output_widget = Output()
    
    def update_plot(change=None):
        with output_widget:
            clear_output(wait=True)
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Determine coloring
            color_by = color_by_widget.value
            
            if color_by == 'None':
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    title=f'{x_metric} vs {y_metric}'
                )
                fig.update_traces(
                    marker=dict(
                        color='blue',
                        opacity=opacity_slider.value,
                        size=point_size_slider.value
                    )
                )
            elif color_by == 'Auto (All columns)':
                filtered_df['color_category'] = filtered_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color='color_category',
                    title=f'{x_metric} vs {y_metric}',
                    labels={'color_category': 'Configuration'}
                )
                fig.update_traces(
                    marker=dict(
                        opacity=opacity_slider.value,
                        size=point_size_slider.value
                    )
                )
            else:
                # Check if the color column is numerical or categorical
                is_numerical = filtered_df[color_by].dtype in ['float64', 'int64']
                
                if is_numerical and color_by not in categorical_cols:
                    # Numerical color
                    fig = px.scatter(
                        filtered_df,
                        x=x_metric,
                        y=y_metric,
                        color=color_by,
                        title=f'{x_metric} vs {y_metric}',
                        color_continuous_scale=color_scale_widget.value,
                        labels={color_by: color_by}
                    )
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        )
                    )
                else:
                    # Categorical color
                    fig = px.scatter(
                        filtered_df,
                        x=x_metric,
                        y=y_metric,
                        color=color_by,
                        title=f'{x_metric} vs {y_metric}',
                        color_discrete_sequence=px.colors.qualitative.Set1,
                        labels={color_by: color_by}
                    )
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        )
                    )
            
            # Apply size encoding if selected
            if size_by_widget.value != 'None':
                size_col = size_by_widget.value
                # Normalize sizes to range [5, 20]
                if size_col in filtered_df.columns:
                    size_vals = filtered_df[size_col].fillna(filtered_df[size_col].median())
                    min_size, max_size = size_vals.min(), size_vals.max()
                    if min_size != max_size:
                        normalized_sizes = 5 + (size_vals - min_size) / (max_size - min_size) * 15
                    else:
                        normalized_sizes = [10] * len(filtered_df)
                    
                    fig.update_traces(
                        marker=dict(
                            size=normalized_sizes,
                            sizemode='area',
                            sizeref=2.*max(normalized_sizes)/(40**2)
                        )
                    )
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout based on toggles
            fig.update_layout(
                plot_bgcolor='white',
                xaxis=dict(
                    title=x_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                yaxis=dict(
                    title=y_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                hovermode='closest',
                height=600,
                showlegend=legend_toggle.value
            )
            
            # Add hover information
            hover_template = "<b>Configuration</b><br>"
            for col in categorical_cols:
                hover_template += f"{col}: %{{customdata[{categorical_cols.index(col)}]}}<br>"
            hover_template += f"<br><b>{x_metric}</b>: %{{x:.4f}}<br>"
            hover_template += f"<b>{y_metric}</b>: %{{y:.4f}}<br>"
            hover_template += "<extra></extra>"
            
            # Add customdata for hover
            fig.update_traces(
                customdata=filtered_df[categorical_cols],
                hovertemplate=hover_template
            )
            
            fig.show()
    
    # Create UI layout
    # Left panel with filters
    filter_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *[HBox([widget]) for widget in filter_widgets.values()]
    ])
    
    # Right panel with styling options
    style_panel = VBox([
        widgets.HTML("<b>Styling:</b>"),
        color_by_widget,
        size_by_widget,
        color_scale_widget,
        opacity_slider,
        point_size_slider,
        legend_toggle,
        grid_toggle
    ])
    
    # Main control panel
    control_panel = HBox([filter_panel, style_panel])
    
    # Add reset button
    reset_button = widgets.Button(description='Reset All', button_style='warning')
    
    def reset_all(b):
        for widget in filter_widgets.values():
            widget.value = []
        color_by_widget.value = 'Auto (All columns)'
        size_by_widget.value = 'None'
        opacity_slider.value = 0.7
        point_size_slider.value = 8
        legend_toggle.value = True
        grid_toggle.value = True
        color_scale_widget.value = 'Viridis'
    
    reset_button.on_click(reset_all)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    size_by_widget.observe(update_plot, names='value')
    opacity_slider.observe(update_plot, names='value')
    point_size_slider.observe(update_plot, names='value')
    legend_toggle.observe(update_plot, names='value')
    grid_toggle.observe(update_plot, names='value')
    color_scale_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display complete interface
    display(VBox([control_panel, reset_button, output_widget]))
    
    return filter_widgets, color_by_widget, size_by_widget


# Helper function to create a static version with dropdown filters using Plotly's built-in features
def create_static_with_filters(df, x_metric, y_metric, signature_flag=False):
    """
    Create a static Plotly figure with dropdown filters (no ipywidgets required)
    """
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Create color category
    df_pivoted['color_category'] = df_pivoted[id_vars].astype(str).agg(' | '.join, axis=1)
    
    # Create figure
    fig = px.scatter(
        df_pivoted,
        x=x_metric,
        y=y_metric,
        color='color_category',
        title=f'{x_metric} vs {y_metric}',
        labels={'color_category': 'Configuration'}
    )
    
    # Add filters as dropdown menus
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    
    buttons = []
    for col in categorical_cols:
        unique_vals = ['All'] + sorted(df_pivoted[col].dropna().unique())
        for val in unique_vals:
            buttons.append(
                dict(
                    method='restyle',
                    label=f'{col}: {val}',
                    args=[{'visible': [val == 'All' or df_pivoted[col].iloc[i] == val for i in range(len(df_pivoted))]}]
                )
            )
    
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=buttons[:10],  # Limit to first 10 for performance
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.1,
                xanchor="left",
                y=1.1,
                yanchor="top"
            ),
        ]
    )
    
    # Add zero lines if requested
    if signature_flag:
        fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
        fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
    
    fig.update_layout(
        plot_bgcolor='white',
        xaxis=dict(gridcolor='lightgray', showgrid=True, zeroline=False),
        yaxis=dict(gridcolor='lightgray', showgrid=True, zeroline=False),
        height=600
    )
    
    return fig


# Example usage:
# For interactive version (requires ipywidgets):
# create_interactive_metric_scatter(df, 'accuracy', 'loss', signature_flag=True)

# For advanced version with more controls:
# create_advanced_interactive_scatter(df, 'accuracy', 'loss', signature_flag=True)

# For static version with dropdowns (no ipywidgets needed):
# fig = create_static_with_filters(df, 'accuracy', 'loss', signature_flag=True)
# fig.show()

In [14]:
total_get_weights = (comp_metrics[comp_metrics.metric == 'get_weights_avg_time'].drop('metric', axis=1).set_index([c for c in comp_metrics if c not in ('value', 'metric')]) *
  comp_metrics[comp_metrics.metric == 'get_weights_avg_time'].drop('metric', axis=1).set_index([c for c in comp_metrics if c not in ('value', 'metric')]))

total_get_indexers = comp_metrics[comp_metrics.metric == 'get_indexers_total_time'].drop('metric', axis=1).set_index([c for c in comp_metrics if c not in ('value', 'metric')])

total_overhead = (total_get_weights + total_get_indexers) /  comp_metrics[comp_metrics.metric == 'train_time'].drop('metric', axis=1).set_index([c for c in comp_metrics if c not in ('value', 'metric')]) * 100
total_overhead['metric'] = 'sketch_proportion'

In [15]:
sketched_trees = comp_metrics[comp_metrics.metric == 'get_indexers_total_time'].drop('metric', axis=1).set_index([c for c in comp_metrics if c not in ('value', 'metric')])

sketched_trees = (sketched_trees) /  comp_metrics[comp_metrics.metric == 'ntrees'].drop('metric', axis=1).set_index([c for c in comp_metrics if c not in ('value', 'metric')]) * 100
sketched_trees['metric'] = 'sketched_tree_proportion'

In [16]:
enriched = pd.concat([
    comp_metrics, 
    total_overhead.reset_index(),
    sketched_trees.reset_index()
], axis=0)

In [20]:
enriched['experiment'] = enriched['experiment'].map(lambda x: 'sigmoid' if x.startswith('sig') else 'linear')

In [28]:
filter_df(enriched, {'lr': 0.1, 'stabilization_threshold': 1.0})

,dataset,sketch_method,sketch_outputs,smoothing_alpha,stabilization_threshold,subsample,lr,metric,value,experiment
17,cifar10,topk,1,0.7,1.0,0.05,0.1,f1,0.269058,sigmoid
19,cifar10,topk,1,0.7,1.0,0.05,0.1,get_indexers_calls,17.000000,sigmoid
20,cifar10,topk,1,0.7,1.0,0.05,0.1,get_indexers_total_time,12.040806,sigmoid
21,cifar10,topk,1,0.7,1.0,0.05,0.1,get_weights_avg_time,0.356189,sigmoid
22,cifar10,topk,1,0.7,1.0,0.05,0.1,get_weights_calls,17.000000,sigmoid
...,...,...,...,...,...,...,...,...,...,...
611,cifar10,topk,10,0.9,1.0,0.50,0.1,sketched_tree_proportion,50.272019,sigmoid
614,cifar10,topk,10,0.9,1.0,0.75,0.1,sketched_tree_proportion,46.610204,linear
615,cifar10,topk,10,0.9,1.0,0.75,0.1,sketched_tree_proportion,43.165266,sigmoid
618,cifar10,topk,10,0.9,1.0,1.00,0.1,sketched_tree_proportion,42.344435,linear


In [ ]:
create_advanced_interactive_scatter(enriched, 'train_time', 'sketch_proportion')

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='250px'), options=('cifar10',), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='250px'), options=('topk',), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='250px'), options=(np.int64(1), np.int64(2), np.int64(5), np.int64(7), np.int64(10)), style=DescriptionStyle(description_width='initial'), value=()),
  'smoothing_alpha': SelectMultiple(description='smoothing_alpha', layout=Layout(width='250px'), options=(np.float64(0.7), np.float64(0.8), np.float64(0.9)), style=DescriptionStyle(description_width='initial'), value=()),
  'stabilization_threshold': SelectMultiple(description='stabilization_t', layout=Layout(width='250px'), options=(np.float64(1.0), np.float64(2.0)), style=DescriptionStyle(description_width='initial')

In [ ]:
def create_interactive_seaborn_metric_scatter(
    df,
    metric_a,
    metric_b,
    cluster_by,
    filters=None,
    signature_flag=True,
    compare_label='sketch proportion vs train time',
    out_dir='.',
    save_image=True,
):
    import re
    from pathlib import Path
    import numpy as np
    import plotly.express as px
 
    df_local = df.copy()
 
    if filters:
        for col, val in filters.items():
            if col not in df_local.columns:
                continue
            if isinstance(val, np.ndarray):
                allowed = np.ravel(val).tolist()
                df_local = df_local[df_local[col].isin(allowed)]
            elif isinstance(val, (list, tuple, set)):
                df_local = df_local[df_local[col].isin(list(val))]
            else:
                df_local = df_local[df_local[col] == val]
 
    id_vars = [c for c in df_local.columns if c not in ['metric', 'value']]
    df_pivoted =df_local.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
 
    missing_cols = [c for c in [metric_a, metric_b, cluster_by] if c not in df_pivoted.columns]
    if missing_cols:
        raise KeyError(f"Missing columns after pivot: {missing_cols}")
 
    df_plot = df_pivoted.dropna(subset=[metric_a, metric_b]).copy()
    if df_plot.empty:
        print('No rows left after filtering/pivot for selected metrics.')
        return None
 
    # Make legend cleaner and deterministic
    df_plot[cluster_by] = df_plot[cluster_by].astype(str)
    hover_cols = [c for c in id_vars if c != cluster_by]
 
    title = f"{metric_a} vs {metric_b} | {compare_label} | clusters: {cluster_by}"
    fig = px.scatter(
        df_plot,
        x=metric_a,
        y=metric_b,
        color=cluster_by,
        hover_data=hover_cols,
        opacity=0.82,
        template='simple_white',
        title=title,
    )
 
    fig.update_traces(
        marker=dict(size=9, line=dict(width=0.8, color='rgba(20,20,20,0.55)'))
    )
 
    if signature_flag:
        fig.add_vline(x=0, line_width=1.2, line_dash='dash', line_color='gray', opacity=0.75)
        fig.add_hline(y=0, line_width=1.2, line_dash='dash', line_color='gray', opacity=0.75)
 
    fig.update_layout(
        legend_title_text=cluster_by,
        legend_title_font=dict(size=20, family='Arial', weight='bold'),
        legend_font=dict(size=20, family='Arial'),
        xaxis_title=metric_a,
        xaxis_title_font=dict(size=20, family='Arial', weight='bold'),
        yaxis_title=metric_b,
        yaxis_title_font=dict(size=20, family='Arial', weight='bold'),
        title_font=dict(size=24, family='Arial', weight='bold'),
        height=700,
        width=1200,
    )
    fig.update_xaxes(
        showgrid=True, 
        gridcolor='lightgray',
        title_font=dict(size=20),
        tickfont=dict(size=20, family='Arial')
    )
    fig.update_yaxes(
        showgrid=True, 
        gridcolor='lightgray',
        title_font=dict(size=20),
        tickfont=dict(size=20, family='Arial')
    )
 
    if save_image:
        save_dir = Path(out_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
 
        safe = lambda s: re.sub(r'[^A-Za-z0-9._-]+', '_', str(s)).strip('_')
        file_stem = f"{safe(metric_a)}_vs_{safe(metric_b)}__{safe(compare_label)}__by_{safe(cluster_by)}"
        img_path = save_dir / f"{file_stem}.png"
 
        fig.write_image(img_path, scale=2)
        print(f"Saved high-res image: {img_path}")
 
    fig.show()
    # return fig
 

In [36]:
prop = filter_df(enriched, {'lr': 0.1, 'stabilization_threshold': 1.0})
prop = prop[prop.metric.isin(['sketch_proportion', 'train_time'])]

prop.groupby(['metric', 'experiment', ]).agg({'value': ['mean', 'std']})

value             
                                     mean          std
metric            experiment                          
sketch_proportion linear         1.248961     0.353251
                  sigmoid        1.333078     0.362106
train_time        linear      4371.702263  1148.701684
                  sigmoid     3252.911036  1353.562081

In [39]:
import pandas as pd

# 1. Compute mean and std
agg = prop.groupby(['metric', 'experiment']).agg(
    mean=('value', 'mean'),
    std=('value', 'std')
).reset_index()

# 2. Create formatted column
agg['mean ± std'] = agg.apply(
    lambda x: f"{x['mean']:.2f} ± {x['std']:.2f}", axis=1
)

# 3. Pivot to metric × experiment
pivot = agg.pivot(
    index='metric', 
    columns='experiment', 
    values='mean ± std'
)

(pivot).to_latex()

'\\begin{tabular}{lll}\n\\toprule\nexperiment & linear & sigmoid \\\\\nmetric &  &  \\\\\n\\midrule\nsketch_proportion & 1.25 ± 0.35 & 1.33 ± 0.36 \\\\\ntrain_time & 4371.70 ± 1148.70 & 3252.91 ± 1353.56 \\\\\n\\bottomrule\n\\end{tabular}\n'

In [ ]:
create_interactive_seaborn_metric_scatter(
    (enriched), 'train_time', 'sketch_proportion', 'experiment', filters={'lr': 0.1, 'smoothing_alpha': 0.9} 
)

Saved high-res image: train_time_vs_sketch_proportion__sketch_proportion_vs_train_time__by_experiment.png


In [25]:
enriched.metric.unique()

array(['f1', 'get_indexers_calls', 'get_indexers_total_time',
       'get_weights_avg_time', 'get_weights_calls', 'inference_time',
       'mean_leaves', 'mean_nodes', 'ntrees', 'train_time',
       'sketch_proportion', 'sketched_tree_proportion'], dtype=object)

In [ ]:
import pandas as pd

# 1. Compute mean and std
agg = prop.groupby(['metric', 'experiment']).agg(
    mean=('value', 'mean'),
    std=('value', 'std')
).reset_index()

# 2. Create formatted column
agg['mean ± std'] = agg.apply(
    lambda x: f"{x['mean']:.4f} ± {x['std']:.4f}", axis=1
)

# 3. Pivot to metric × experiment
pivot = agg.pivot(
    index='metric', 
    columns='experiment', 
    values='mean ± std'
)

print(pivot)

,dataset,sketch_method,sketch_outputs,smoothing_alpha,stabilization_threshold,subsample,lr,metric,value,experiment
1,cifar10,topk,1,0.7,1.0,0.05,0.005,f1,0.211457,sigmoid
3,cifar10,topk,1,0.7,1.0,0.05,0.005,get_indexers_calls,118.800000,sigmoid
4,cifar10,topk,1,0.7,1.0,0.05,0.005,get_indexers_total_time,81.550131,sigmoid
5,cifar10,topk,1,0.7,1.0,0.05,0.005,get_weights_avg_time,0.338965,sigmoid
6,cifar10,topk,1,0.7,1.0,0.05,0.005,get_weights_calls,118.800000,sigmoid
...,...,...,...,...,...,...,...,...,...,...
645,cifar10,topk,10,NaN,NaN,0.50,0.100,sketched_tree_proportion,NaN,linear
646,cifar10,topk,10,NaN,NaN,0.75,0.005,sketched_tree_proportion,NaN,linear
647,cifar10,topk,10,NaN,NaN,0.75,0.100,sketched_tree_proportion,NaN,linear
648,cifar10,topk,10,NaN,NaN,1.00,0.005,sketched_tree_proportion,NaN,linear


In [ ]:
create_advanced_interactive_scatter(enriched,
                                     'train_time', 'sketch_proportion')

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='250px'), options=('cifar10',), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='250px'), options=('topk',), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='250px'), options=(np.int64(1), np.int64(2), np.int64(5), np.int64(7), np.int64(10)), style=DescriptionStyle(description_width='initial'), value=()),
  'smoothing_alpha': SelectMultiple(description='smoothing_alpha', layout=Layout(width='250px'), options=(np.float64(0.7), np.float64(0.8), np.float64(0.9)), style=DescriptionStyle(description_width='initial'), value=()),
  'stabilization_threshold': SelectMultiple(description='stabilization_t', layout=Layout(width='250px'), options=(np.float64(1.0), np.float64(2.0)), style=DescriptionStyle(description_width='initial')

In [20]:
create_advanced_interactive_scatter(enriched,
                                     'ntrees', 'train_time')

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='250px'), options=('cifar10',), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='250px'), options=('topk',), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='250px'), options=(np.int64(1), np.int64(2), np.int64(5), np.int64(7), np.int64(10)), style=DescriptionStyle(description_width='initial'), value=()),
  'smoothing_alpha': SelectMultiple(description='smoothing_alpha', layout=Layout(width='250px'), options=(np.float64(0.7), np.float64(0.8), np.float64(0.9)), style=DescriptionStyle(description_width='initial'), value=()),
  'stabilization_threshold': SelectMultiple(description='stabilization_t', layout=Layout(width='250px'), options=(np.float64(1.0), np.float64(2.0)), style=DescriptionStyle(description_width='initial')

In [21]:
create_advanced_interactive_scatter(enriched,
                                     'ntrees', 'mean_leaves')

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='250px'), options=('cifar10',), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='250px'), options=('topk',), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='250px'), options=(np.int64(1), np.int64(2), np.int64(5), np.int64(7), np.int64(10)), style=DescriptionStyle(description_width='initial'), value=()),
  'smoothing_alpha': SelectMultiple(description='smoothing_alpha', layout=Layout(width='250px'), options=(np.float64(0.7), np.float64(0.8), np.float64(0.9)), style=DescriptionStyle(description_width='initial'), value=()),
  'stabilization_threshold': SelectMultiple(description='stabilization_t', layout=Layout(width='250px'), options=(np.float64(1.0), np.float64(2.0)), style=DescriptionStyle(description_width='initial')